In [1]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
from statsmodels.stats.sandwich_covariance import cov_hac
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from arch import arch_model
from arch.univariate import ARX
from scipy.stats import norm

### Import data

In [2]:
path = os.path.abspath('E:/RA/Geert/task1.py')
dir_path = os.path.dirname(path)
os.chdir(dir_path)
excel_file = pd.ExcelFile('Aggregate_CPI_inflation_20230513.xls')
sheet_quarter = excel_file.sheet_names[0]
sheet_month = excel_file.sheet_names[1]
#quarterly and monthly aggregate CPI data (deseasonalized). The full sample is 1947-2022
data_quarter = excel_file.parse(sheet_quarter, skiprows=2)
data_month = excel_file.parse(sheet_month, skiprows=2)
data_quarter.index = pd.to_datetime(data_quarter['Year'].astype(str) + '-Q' + data_quarter['Quarter'].astype(str))
data_month.index = pd.to_datetime(data_month[['Year', 'Month']].assign(day=1))
data_quarter.columns = ['Year', 'Quarter', 'Price index', 'Inflation', 'Forecasted inflation', 'Inflation shock']
data_month.columns = ['Year', 'Month', 'Price index', 'Inflation', 'Forecasted inflation', 'Inflation shock']
sample_data = data_quarter[data_quarter['Year']>1969]

In [3]:
sample_data['Inflation_lag_1'] =  sample_data['Inflation'].shift(1)
sample_data['Inflation_lag_2'] =  sample_data['Inflation'].shift(2)
sample_data['Forecasted_inflation_lag_1'] =  sample_data['Forecasted inflation'].shift(1)
sample_data = sample_data.dropna()

C:\Users\ADMINI~1\AppData\Local\Temp/ipykernel_11168/234630771.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sample_data['Inflation_lag_1'] =  sample_data['Inflation'].shift(1)
C:\Users\ADMINI~1\AppData\Local\Temp/ipykernel_11168/234630771.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sample_data['Inflation_lag_2'] =  sample_data['Inflation'].shift(2)
C:\Users\ADMINI~1\AppData\Local\Temp/ipykernel_11168/234630771.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice fro

# Summary Statistics

## Model Selection

 ### GARCH model specification

GJR-GARCH model specification:
$$
\sigma_{t}^{2}=\omega  + \sum_{i=1}^{p}\left(\alpha_{i}\epsilon_{t-i}^{2}
        +\gamma_{i}\epsilon_{t-i}^{2}
        I\left[\epsilon_{t-i}<0\right]\right)+\sum_{k=1}^{q}\beta_{k}\sigma_{t-k}^{2}
$$


In the previous results, we know GJR-GARCH(2,1) outperforms in every mean model case. So I compare the performance of different error distributions while using GJR-GARCH(2,1) as volatility process.

#### standardized Student's t distribution

The log-likelihood of a single data point $\epsilon_t$ with conditional standard error $\sigma_t$ is
$$
\ln\Gamma\left(\frac{\nu+1}{2}\right)   -\ln\Gamma\left(\frac{\nu}{2}\right)
 -\frac{1}{2}\ln(\pi\left(\nu-2\right)\sigma_t^{2})
            -\frac{\nu+1}{2}\ln(1+\epsilon_t^{2}/(\sigma_t^{2}(\nu-2)))
$$

#### standardized Skew Student's t distribution

 The log-likelihood of a single data point $\epsilon_t$ with conditional standard error $\sigma_t$ is
$$
\ln\left[\frac{bc}{\sigma_t}\left(1+\frac{1}{\eta-2}
\left(\frac{a+b\epsilon_t/\sigma_t}
{1+sgn(\epsilon_t/\sigma_t+a/b)\lambda}\right)^{2}\right)^{-\left(\eta+1\right)/2}\right]
$$

The Standardized Skewed Student's distribution  takes two parameters,
    $\eta$ and $\lambda$. $\eta$ controls the tail shape
    and is similar to the shape parameter in a Standardized Student's t.
    $\lambda$ controls the skewness. When $\lambda=0$ the
    distribution is identical to a standardized Student's t.

#### Generalized Error Distribution

The log-likelihood of a single data point $\epsilon_t$  with conditional standard error $\sigma_t$ is
$$
\ln\nu-\ln c-\ln\Gamma(\frac{1}{\nu})-(1+\frac{1}{\nu})\ln2
            -\frac{1}{2}\ln\sigma_t^{2}
            -\frac{1}{2}\left|\frac{\epsilon_t}{c\sigma_t}\right|^{\nu}
$$
where $\Gamma$ is the gamma function and $\ln c$ is
$$
\ln c=\frac{1}{2}\left(\frac{-2}{\nu}\ln2+\ln\Gamma(\frac{1}{\nu})
            -\ln\Gamma(\frac{3}{\nu})\right)
$$

#### Mixture of normal distributions

The log-likelihood of a single data point $\epsilon_t$ with conditional standard error $\sigma_t$ is
$$
 \ln \frac{1}{\sigma_t} \left[
 p_1  \frac{1}{\sqrt{ 2\pi\sigma_1^2} } exp\{ -\frac{(z_t-\mu_1)^2}{2\sigma_1^2} \}
+(1-p_1)\frac{1}{\sqrt{ 2\pi\sigma_2^2} } exp\{ -\frac{(z_t-\mu_2)^2}{2\sigma_2^2} \} \right]
$$,
where $z_t = \frac{\epsilon_t}{\sigma_t}$

### Mean model and Variance model are jointly estimated. 

There is a standard package estimating Gaussian Mixture Model. So I use the standardized residuals from GJR-Garch to estimate parameters of mixture of 2 normals, and then use those parameters as starting points. This method is really like what you said in the last email by using moments. 

In [5]:
def GarchFamilyResults(Y,X=None,mean='Zero',lags=None,cov = 'robust'):
    options = {'maxiter': 1000}

    gjr_normal = arch_model(y=Y,x=X, mean=mean, vol='GARCH',p=2, o=2,q=1,dist='normal',lags=lags).fit(disp='off',cov_type=cov,options=options)
    gjr_studentst = arch_model(y=Y,x=X, mean=mean, vol='GARCH',p=2, o=2,q=1,dist='studentst',lags=lags).fit(disp='off',cov_type=cov,options=options)
    gjr_skewstudent = arch_model(y=Y,x=X, mean=mean, vol='GARCH',p=2, o=2,q=1,dist='skewstudent',lags=lags).fit(disp='off',cov_type=cov,options=options)
    gjr_generalized = arch_model(y=Y,x=X, mean=mean, vol='GARCH',p=2, o=2,q=1,dist='generalized error',lags=lags).fit(disp='off',cov_type=cov,options=options)
    
    stdresid = gjr_normal.resid / gjr_normal.conditional_volatility
    gmm = GaussianMixture(n_components=2).fit(np.array(stdresid.dropna()).reshape(-1,1))
    starting_values = np.concatenate( (np.array(gjr_normal.params), np.array([gmm.weights_[0],gmm.means_ [0][0], gmm.covariances_[0][0][0]]) )   )
    
    sparch21 = arch_model(y=Y,x=X, mean=mean, vol='GARCH',p=2, o=2, q=1,lags=lags)
    sparch21.distribution = MixNormal()
    gjr_MixNormal=sparch21.fit(disp='off',cov_type=cov,options=options,starting_values=starting_values   )
    print('\nGJR-GARCH(2,1) model')
    print('\t \t \t AIC: \t \t \t  BIC',
          '\nNormal:            ',gjr_normal.aic,'\t',gjr_normal.bic,
          '\nStudents T:        ',gjr_studentst.aic,'\t',gjr_studentst.bic,
          '\nSkew Student T:    ',gjr_skewstudent.aic,'\t',gjr_skewstudent.bic,
          '\nGeneralized Error: ',gjr_generalized.aic,'\t',gjr_generalized.bic,
          '\nMixture Normal:    ',gjr_MixNormal.aic,'\t',gjr_MixNormal.bic,
         )


## Mean Model

$
\pi (t) =  fc(t-1) + \epsilon (t)
$

In [6]:
GarchFamilyResults(Y = sample_data['Inflation shock'],mean='Zero')

NameError: name 'GaussianMixture' is not defined

$
\pi (t) = c +\rho \pi (t-1)   + \phi fc(t-1) + \epsilon (t)
$

In [39]:
GarchFamilyResults(Y = sample_data['Inflation'],X=sample_data['Forecasted inflation'],mean='ARX',lags=[1])


GJR-GARCH(2,1) model
	 	 	 AIC: 	 	 	  BIC 
Normal:             359.4710395915945 	 389.46550873098283 
Students T:         343.1431206566 	 376.4703085892537 
Skew Student T:     345.1311935562092 	 381.79110028212824 
Generalized Error:  345.1925329050945 	 378.5197208377482 
Mixture Normal:     4068.355658769029 	 4108.348284288213


D:\anaconda\lib\site-packages\arch\univariate\base.py:704: StartingValueWarning: Starting values do not satisfy the parameter constraints in the model.  The
provided starting values will be ignored.

  warnings.warn(starting_value_warning, StartingValueWarning)
C:\Users\ADMINI~1\AppData\Local\Temp/ipykernel_11520/1115095375.py:102: RuntimeWarning: invalid value encountered in sqrt
  sigma_2 = sqrt(p1/(p1-1)*sigma_1**2 - p1/(p1-1)**2 * u1**2 + 1/(1-p1))
C:\Users\ADMINI~1\AppData\Local\Temp/ipykernel_11520/1115095375.py:104: RuntimeWarning: divide by zero encountered in log
  lls = log( 1/sqrt(sigma2) * ( p1/sqrt(2*pi*sigma_1**2) *  exp(-(resids/sqrt(sigma2)-u1)**2/(2*sigma_1**2))   +   p2/sqrt(2*pi*sigma_2**2) *  exp(-(resids/sqrt(sigma2)-u2)**2/(2*sigma_2**2))        )     )
C:\Users\ADMINI~1\AppData\Local\Temp/ipykernel_11520/1115095375.py:101: RuntimeWarning: divide by zero encountered in double_scalars
  u2 = p1/(p1-1) *u1
C:\Users\ADMINI~1\AppData\Local\Temp/ipykernel_11520/1115095

$
\pi (t) = c +\rho_1 \pi (t-1)  +\rho_2 \pi (t-2)  + \phi fc(t-1) + \epsilon (t)
$

In [40]:
GarchFamilyResults(Y = sample_data['Inflation'],X=sample_data['Forecasted inflation'],mean='ARX',lags=[1,2])

D:\anaconda\lib\site-packages\arch\univariate\base.py:704: StartingValueWarning: Starting values do not satisfy the parameter constraints in the model.  The
provided starting values will be ignored.

  warnings.warn(starting_value_warning, StartingValueWarning)
C:\Users\ADMINI~1\AppData\Local\Temp/ipykernel_11520/1115095375.py:102: RuntimeWarning: invalid value encountered in sqrt
  sigma_2 = sqrt(p1/(p1-1)*sigma_1**2 - p1/(p1-1)**2 * u1**2 + 1/(1-p1))
C:\Users\ADMINI~1\AppData\Local\Temp/ipykernel_11520/1115095375.py:104: RuntimeWarning: divide by zero encountered in log
  lls = log( 1/sqrt(sigma2) * ( p1/sqrt(2*pi*sigma_1**2) *  exp(-(resids/sqrt(sigma2)-u1)**2/(2*sigma_1**2))   +   p2/sqrt(2*pi*sigma_2**2) *  exp(-(resids/sqrt(sigma2)-u2)**2/(2*sigma_2**2))        )     )
C:\Users\ADMINI~1\AppData\Local\Temp/ipykernel_11520/1115095375.py:101: RuntimeWarning: divide by zero encountered in double_scalars
  u2 = p1/(p1-1) *u1
C:\Users\ADMINI~1\AppData\Local\Temp/ipykernel_11520/1115095


GJR-GARCH(2,1) model
	 	 	 AIC: 	 	 	  BIC 
Normal:             360.22102427875654 	 393.4997859666523 
Students T:         343.5443712269164 	 380.15100908360176 
Skew Student T:     345.5440818520591 	 385.4785958775341 
Generalized Error:  345.53270988301125 	 382.13934773969663 
Mixture Normal:     366.22102389416403 	 409.4834140884286


D:\anaconda\lib\site-packages\arch\univariate\base.py:756: ConvergenceWarning: The optimizer returned code 8. The message is:
Positive directional derivative for linesearch
See scipy.optimize.fmin_slsqp for code meaning.

  warnings.warn(


$
\pi (t) = c +\rho_1 \pi (t-1)  +\rho_2 \pi (t-2)  + \phi_1 fc(t-1) + \phi_2 fc(t-2) + \epsilon (t)
$

In [41]:
GarchFamilyResults(Y = sample_data['Inflation'],X=sample_data[['Forecasted inflation','Forecasted_inflation_lag_1']],mean='ARX',lags=[1,2])

D:\anaconda\lib\site-packages\arch\univariate\base.py:704: StartingValueWarning: Starting values do not satisfy the parameter constraints in the model.  The
provided starting values will be ignored.

  warnings.warn(starting_value_warning, StartingValueWarning)
C:\Users\ADMINI~1\AppData\Local\Temp/ipykernel_11520/1115095375.py:102: RuntimeWarning: invalid value encountered in sqrt
  sigma_2 = sqrt(p1/(p1-1)*sigma_1**2 - p1/(p1-1)**2 * u1**2 + 1/(1-p1))
C:\Users\ADMINI~1\AppData\Local\Temp/ipykernel_11520/1115095375.py:104: RuntimeWarning: divide by zero encountered in log
  lls = log( 1/sqrt(sigma2) * ( p1/sqrt(2*pi*sigma_1**2) *  exp(-(resids/sqrt(sigma2)-u1)**2/(2*sigma_1**2))   +   p2/sqrt(2*pi*sigma_2**2) *  exp(-(resids/sqrt(sigma2)-u2)**2/(2*sigma_2**2))        )     )
C:\Users\ADMINI~1\AppData\Local\Temp/ipykernel_11520/1115095375.py:101: RuntimeWarning: divide by zero encountered in double_scalars
  u2 = p1/(p1-1) *u1
C:\Users\ADMINI~1\AppData\Local\Temp/ipykernel_11520/1115095


GJR-GARCH(2,1) model
	 	 	 AIC: 	 	 	  BIC 
Normal:             362.15768850699413 	 398.7643263636795 
Students T:         345.4535993840781 	 385.38811340955306 
Skew Student T:     347.43805641909074 	 390.7004466133553 
Generalized Error:  347.46746827688594 	 387.4019823023609 
Mixture Normal:     356.6727187637222 	 403.2629851267763


D:\anaconda\lib\site-packages\arch\univariate\base.py:756: ConvergenceWarning: The optimizer returned code 8. The message is:
Positive directional derivative for linesearch
See scipy.optimize.fmin_slsqp for code meaning.

  warnings.warn(


In [13]:
from __future__ import annotations
from arch.univariate.distribution import Distribution
from abc import ABCMeta, abstractmethod
from collections.abc import Sequence
from typing import Callable
import warnings

from numpy import (
    abs,    array,    asarray,    empty,    exp,    int64,    
    integer,    isscalar,    log,    nan,    ndarray, exp,
    ones_like,    pi,    sign,    sqrt,    sum,)
from numpy.random import Generator, RandomState, default_rng
from scipy.special import comb, gamma, gammainc, gammaincc, gammaln
import scipy.stats as stats
from scipy.optimize import bisect

from arch.typing import ArrayLike, ArrayLike1D, Float64Array
from arch.utility.array import AbstractDocStringInheritor, ensure1d
class MixNormal(Distribution, metaclass=AbstractDocStringInheritor):
    """
    Mixture of two Normal distributions for use with SPARCH model

    Parameters
    ----------
    random_state : RandomState, optional
        .. deprecated:: 5.0

           random_state is deprecated. Use seed instead.

    seed : {int, Generator, RandomState}, optional
        Random number generator instance or int to use. Set to ensure
        reproducibility. If using an int, the argument is passed to
        ``np.random.default_rng``.  If not provided, ``default_rng``
        is used with system-provided entropy.
    """

    def __init__(
        self,
        random_state: RandomState | None = None,
        *,
        seed: None | int | RandomState | Generator = None,
    ) -> None:
        super().__init__(random_state=random_state, seed=seed)
        self._name = "Mixture of two Normal distributions"
        self.num_params: int = 3  

    def constraints(self) -> tuple[Float64Array, Float64Array]:
        return array([[1, 0, 0], [-1, 0, 0], [0, 1,0], [0, -1,0] , [0,0,1],[0,0,-1]]), array([0, 1, -100,100,0.001, 10000])

    def bounds(self, resids: Float64Array) -> list[tuple[float, float]]:
        """
        Bounds of parameters:
        p1: (0,1)
        u1: (-100,100)
        sigma1:(0.0001*var(resids),1000*var(resids))
        """
        return [(0, 1),(-100,100),(0.001, 10000)]

    def loglikelihood(
        self,
        parameters: Sequence[float] | ArrayLike1D,
        resids: ArrayLike,
        sigma2: ArrayLike,
        individual: bool = False,
    ) -> float | Float64Array:
        r"""Computes the log-likelihood of assuming residuals are mixture normally
        distributed, conditional on the variance

        Parameters
        ----------
        parameters : ndarray
            Parameters of the first normal distribution: p1,u1,sigma1. Second one can be calculated by restrictions.
        resids  : ndarray
            The residuals to use in the log-likelihood calculation
        sigma2 : ndarray
            Conditional variances of resids
        individual : bool, optional
            Flag indicating whether to return the vector of individual log
            likelihoods (True) or the sum (False)

        Returns
        -------
        ll : float
            The log-likelihood

        Notes
        -----
        The log-likelihood of a single data point x is

        .. math::

            \ln f\left(x\right)=
            \ln \frac{1}{\sqrt{h_t}} \left[
                 p_1  \frac{1}{\sqrt{ 2\pi\sigma_1^2} } exp\{ -\frac{(x-\mu_1)^2}{2\sigma_1^2} \}
                +(1-p_1)\frac{1}{\sqrt{ 2\pi\sigma_2^2} } exp\{ -\frac{(x-\mu_2)^2}{2\sigma_2^2} \} \right]

        """
        parameters = asarray(parameters, dtype=float)
        p1, u1, sigma_1 = parameters
        p2 = 1- p1
        u2 = p1/(p1-1) *u1
        sigma_2 = sqrt(p1/(p1-1)*sigma_1**2 - p1/(p1-1)**2 * u1**2 + 1/(1-p1))
        
        lls = log( 1/sqrt(sigma2) * ( p1/sqrt(2*pi*sigma_1**2) *  exp(-(resids/sqrt(sigma2)-u1)**2/(2*sigma_1**2))   +   p2/sqrt(2*pi*sigma_2**2) *  exp(-(resids/sqrt(sigma2)-u2)**2/(2*sigma_2**2))        )     ) 
        if individual:
            return lls
        else:
            return sum(lls)

    def starting_values(self, std_resid: Float64Array) -> Float64Array:
        """
        Starting values of parameters
        """
        #gmm = GaussianMixture(n_components=2).fit(std_resid.reshape(-1,1))
        #return array([gmm.weights_[0],gmm.means_ [0][0], gmm.covariances_[0][0][0]])
        
        Weights :  [0.54080114 0.45919886]
    Mean :  [-0.07289659  0.50798397]
    Variance :  [0.89342082 0.98601448]
        
        return array([0.54,0,0.7])
    
    def _simulator(self, size: int | tuple[int, ...]) -> Float64Array:
        assert self._parameters is not None
        p1, u1, sigma_1 = self._parameters
        p2 = 1- p1
        u2 = p1/(p1-1) *u1
        sigma_2 = sqrt(p1/(p1-1)*sigma_1**2 - p1/(p1-1)**2 * u1**2 + 1/(1-p1))
        
        Z = random.choice([0, 1], p=[1 - p1, p1])
        return sigma_1**Z*sigma_2**(1-Z)*self._generator.standard_normal(size) + u1*Z+ u2*(1-Z)

    def simulate(
        self, parameters: int | float | Sequence[float | int] | ArrayLike1D
    ) -> Callable[[int | tuple[int, ...]], Float64Array]:
        parameters = ensure1d(parameters, "parameters", False)
        self._parameters = asarray(parameters, dtype=float)
        return self._simulator

    def parameter_names(self) -> list[str]:
        return ['p_1','mu_1','sigma_1']

    def cdf(
        self,
        resids: Sequence[float] | ArrayLike1D,
        parameters: None | Sequence[float] | ArrayLike1D = None,
    ) -> Float64Array:
        self._check_constraints(parameters)
        
        parameters = asarray(parameters, dtype=float)
        p1, u1, sigma_1 = parameters
        p2 = 1- p1
        u2 = p1/(p1-1) *u1
        sigma_2 = sqrt(p1/(p1-1)*sigma_1**2 - p1/(p1-1)**2 * u1**2 + 1/(1-p1))
        
        return p1*stats.norm.cdf(asarray((resids-u1)/sigma_1)  ) + p2*stats.norm.cdf(asarray((resids-u2)/sigma_2)  )

    def ppf(
        self,
        pits: float | Sequence[float] | ArrayLike1D,
        parameters: None | Sequence[float] | ArrayLike1D = None,
    ) -> Float64Array:
        self._check_constraints(parameters)
        parameters = asarray(parameters, dtype=float)
        p1, u1, sigma_1 = parameters
        p2 = 1- p1
        u2 = p1/(p1-1) *u1
        sigma_2 = sqrt(p1/(p1-1)*sigma_1**2 - p1/(p1-1)**2 * u1**2 + 1/(1-p1)) 
        scalar = isscalar(pits)
        if scalar:
            pits = array([pits])
        else:
            pits = asarray(pits)
            
        def inverse_cdf(cdf, target_p, lower_bound=-100, upper_bound=100):
            def root_func(x):
                return cdf(x,parameters) - target_p
            return bisect(root_func, lower_bound, upper_bound)     
        
        ppf = inverse_cdf(cdf, pits, lower_bound=-100, upper_bound=100)

        if scalar:
            return ppf[0]
        else:
            return ppf
        
   

    def moment(
        self, n: int, parameters: None | Sequence[float] | ArrayLike1D = None
    ) -> float:
        if n < 0:
            return nan
        parameters = asarray(parameters, dtype=float)
        p1, u1, sigma_1 = parameters
        p2 = 1- p1
        u2 = p1/(p1-1) *u1
        sigma_2 = sqrt(p1/(p1-1)*sigma_1**2 - p1/(p1-1)**2 * u1**2 + 1/(1-p1))
        
        moment1 = stats.norm.moment(n,loc=u1,scale=sigma_1)
        moment2 = stats.norm.moment(n,loc=u2,scale=sigma_2)
        return p1 * moment1 + p2 * moment2

    def partial_moment(
        self,
        n: int,
        z: float = 0.0,
        parameters: None | Sequence[float] | ArrayLike1D = None,
        num_samples=100000,
    ) -> float:
        
        parameters = asarray(parameters, dtype=float)
        p1, u1, sigma_1 = parameters
        p2 = 1- p1
        u2 = p1/(p1-1) *u1
        sigma_2 = sqrt(p1/(p1-1)*sigma_1**2 - p1/(p1-1)**2 * u1**2 + 1/(1-p1)) 
        
        if n < 0:
            return nan
        elif n == 0:
            return cdf(z,parameters)
        elif n==1:
            return -p1*stats.norm.pdf(z,loc=u1,scale=sigma_1)  -p2*stats.norm.pdf(z,loc=u2,scale=sigma_2)
        else:
            -(z ** (n - 1)) * (stats.norm.pdf(z,loc=u1,scale=sigma_1)+p2*stats.norm.pdf(z,loc=u2,scale=sigma_2)) 
            + (n - 1) * self.partial_moment(  n - 2, z, parameters  )